# Controlling Dobot

In [ ]:
from pathlib import Path
from dobot import Dobot
import numpy as np

# fix permissions possibly:
#   sudo chmod o+rw /dev/ttyACM0
comport = str(next(Path("/dev").glob("ttyACM*")))
dobot = Dobot(comport, sca1000Sensors=True, endEffectorOffset=(50.9, 15.), plot=True, debug=False)
# percentage of maximum joint speed
speed = 0.5
# percentage of maximum joint acceleration
acceleration = 0.5

In [ ]:
# show the current position in world coordinates
curpos = np.array(dobot.pos)
curpos

In [ ]:
np.rad2deg(dobot._kinematics.anglesFromCoordinates(curpos))

In [ ]:
# this should be a no-op which should not fail
dobot.MoveWithSpeed((curpos-np.array((0,0,0)),), speed, acceleration)

In [ ]:
# do a rectangular motion for testing the motion planning which must stop at the edge
#nextPos += (10,0,0)  # adjust for relative movement
dobot.MoveWithSpeed((curpos, curpos + (-2, 0, 0), curpos + (20, 0, 0), curpos + (20, 20, 0), curpos + (20, 22, 0)), speed, acceleration)

In [ ]:
pts = [
    np.array([230.94713508,   0.        ,  94.01584856]),
    np.array([228.98939573,  12.36067977,  94.01584856]),
    np.array([223.30781486,  23.51141009,  94.01584856]),
    np.array([214.45854517,  32.36067977,  94.01584856]),
    np.array([203.30781486,  38.04226065,  94.01584856]),
    np.array([190.94713508,  40.        ,  94.01584856]),
    np.array([178.58645531,  38.04226065,  94.01584856]),
 np.array([167.43572499,  32.36067977,  94.01584856]),
 np.array([158.58645531,  23.51141009,  94.01584856]),
 np.array([152.90487443,  12.36067977,  94.01584856]),
 np.array([1.50947135e+02, 4.89858720e-15, 9.40158486e+01]),
 np.array([152.90487443, -12.36067977,  94.01584856]),
]
dobot.MoveWithSpeed(pts, speed, acceleration)

In [ ]:
dobot.MoveWithSpeed((curpos + (40, 40, 0), curpos + (40, 45, 0), curpos + (40, -40, 0)), speed, acceleration)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def circle_points(center, radius, n=100):
    cx, cy = center[:2]
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    x = cx + radius * np.cos(theta)
    y = cy + radius * np.sin(theta)
    pts = np.column_stack((x, y))
    # combine and repeat the first point at the end
    return np.vstack((pts, pts[0]))

def plot_points(x, y, center):
    plt.figure(figsize=(5,5))
    plt.plot(x, y, marker='o', markersize=4, linestyle='-')   # circle
    plt.scatter(*center, color='red')                    # center
    plt.gca().set_aspect('equal', 'box')                      # equal scaling
    plt.grid(True)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f"Circle center: {tuple(f"{v:.2f}" for v in center[0:2])}, {radius=})")
    plt.show()

In [ ]:
radius = 40
pts = circle_points(curpos, radius, n=20)
plot_points(pts[:,0], pts[:,1], curpos)
zPos = curpos[2] - 9
pts3 = [np.array((*pt, zPos)) for pt in pts]
pts3

In [ ]:
dobot.MoveWithSpeed(pts3, speed, acceleration)

In [ ]:
for pt in pts[:3]:
    print(f"Going to {(*pt, zPos)} ..")
    dobot.MoveWithSpeed((*pt, zPos), speed, acceleration)

In [ ]:
# back to start pos (center)
dobot.MoveWithSpeed(curpos, speed, acceleration)